# 6.5 — treeAggregate Min and Max

**Chapter 6, section 6.5**, and the starting point for **Exercise 9**.

**The question this notebook answers:** what does a distributed algorithm actually look like?

The task is deliberately simple — find the vehicle with the most trips and the vehicle with the
fewest, in one pass — but the structure is the structure of every distributed fit. The notebook
does the job twice: once with `treeAggregate`, and once by hand in plain Python, with the data
split into artificial partitions, so that the two-level shape hidden inside the single call can be
laid out whole and stepped through.

A single-node algorithm is **one loop** over the data. A distributed algorithm is **a per-partition
computation plus an associative combine.** An algorithm that cannot be written in that two-part
form does not distribute well, however elegant its one-loop version.

Source: `Notebooks old/Spark-Example-09.1-treeAggregate-Min-Max-Calculation.ipynb`, modernized and
revised. Runs on a laptop in well under a minute.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile
from pyspark.sql import SparkSession

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-6.5")
         .master("local[4]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Spark", spark.version)

Spark 4.2.0


---

## 1. The data

The small taxi extract. Two identifier columns matter: `medallion` identifies the **vehicle**, and
`hack_license` identifies the **driver**. The same aggregation answers a question about either.

In [2]:
testFile = os.path.join(DATA, "taxi-data-sorted-verysmall-header.csv")
df = (spark.read.format("csv")
      .options(header="true", inferSchema="true", sep=",")
      .load(testFile))

print("rows:", df.count())
df.select("medallion", "hack_license", "trip_time", "trip_distance").show(5, truncate=18)

rows: 10000
+------------------+------------------+---------+-------------+
|         medallion|      hack_license|trip_time|trip_distance|
+------------------+------------------+---------+-------------+
|07290D3599E7A0D...|E7750A37CAB07D0...|      120|         0.44|
|22D70BF00EEB0AD...|3FF2709163DE703...|        0|          0.0|
|0EC22AAF491A8BD...|778C92B26AE78A9...|      120|         0.71|
|1390FB380189DF6...|BE317B986700F63...|      120|         0.48|
|3B4129883A1D05B...|7077F9FD5AD649A...|      120|         0.61|
+------------------+------------------+---------+-------------+
only showing top 5 rows


---

## 2. The two functions

An aggregation needs two functions, and the division of labour between them is the whole lesson.

* **`seqOp(accumulator, record)`** takes an accumulator and **one record**, and returns the
  updated accumulator. It is the per-record work, and it runs **where the record lives**.
* **`combOp(accumulator, accumulator)`** takes **two accumulators** and merges them. It is the
  combination step, and because it is associative the merging can happen in any grouping —
  within partitions first, across them afterward.

The accumulator here is a four-element list, `(max_key, max_val, min_key, min_val)`.

In [3]:
# example of function input (x = ("k1",-1000000,"k2",1000000) , y = ('07', 165))
def seqOp(x, y):
    # res = (max_key, max_val, min_key, min_val)
    res = [0, 0, 0, 0]
    if x[1] > y[1]:
        # if the current x values are bigger than the y then keep them
        res[0] = x[0]
        res[1] = x[1]
    else:
        # y is bigger, so use y as a maximum
        res[0] = y[0]
        res[1] = y[1]
    if x[3] < y[1]:
        # if the current x values are smaller than the y then keep them
        res[2] = x[2]
        res[3] = x[3]
    else:
        # y is smaller, so use y as a minimum
        res[2] = y[0]
        res[3] = y[1]
    return res


# example of function input (x = ('14',217, '49',70 ) , y = ('13',220, '50',70 ))
def combOp(x, y):
    # res = (max_key, max_val, min_key, min_val)
    res = [0, 0, 0, 0]
    if x[1] > y[1]:
        # if the data in the x is bigger, then use x as the result
        res[0] = x[0]
        res[1] = x[1]
    else:
        res[0] = y[0]
        res[1] = y[1]
    if x[3] < y[3]:
        # if the data in the x is smaller, then use x as the result
        res[2] = x[2]
        res[3] = x[3]
    else:
        res[2] = y[2]
        res[3] = y[3]
    return res

Note the asymmetry in the signatures, because it is the point rather than an oversight. `seqOp`
reads `y[1]` as *the record's count*; `combOp` reads `y[3]` as *the other accumulator's minimum*.
The two functions take different second arguments and cannot be interchanged.

### Testing them before trusting them

In [4]:
print(seqOp(["k1", -1000000, "k2", 1000000], ("k3", 5)))   # first record seen
print(seqOp(["k1", 20, "k2", 10], ("k3", 5)))              # a new minimum
print(seqOp(["k1", 20, "k2", 10], ("k3", 25)))             # a new maximum

['k3', 5, 'k3', 5]
['k1', 20, 'k3', 5]
['k3', 25, 'k2', 10]


In [5]:
print(combOp(["k1", 20, "k2", 10], ["k3", 25, "k4", 15]))  # the other side has the maximum
print(combOp(["k1", 20, "k2", 10], ["k3", 25, "k4", 5]))   # ... and the minimum

['k3', 25, 'k2', 10]
['k3', 25, 'k4', 5]


---

## 3. The zero value, and why it has to be an identity

The aggregation is seeded with a zero value whose sentinel entries — $-10^{6}$ as the initial
maximum and $10^{6}$ as the initial minimum — **lose every comparison against real data**.

That is not a convenience. An aggregation's zero value must be an **identity element** for the
combining function: `combOp(zero, acc)` must return `acc` unchanged, for any `acc`. Spark folds
the zero value into each partition independently and is free to apply `combOp` in any grouping, so
a zero value that is not an identity produces an answer that depends on the partitioning.

In [6]:
agg_zero_val = ["k1", -1000000, "k2", 1000000]

# the requirement, checked rather than asserted
for acc in [["a", 7, "b", 1], ["a", -5, "b", -9], ["a", 0, "b", 0]]:
    assert combOp(agg_zero_val, acc) == acc, acc
    assert combOp(acc, agg_zero_val) == acc, acc
    print(f"combOp(zero, {acc}) == {acc}  -- identity holds on both sides")

combOp(zero, ['a', 7, 'b', 1]) == ['a', 7, 'b', 1]  -- identity holds on both sides
combOp(zero, ['a', -5, 'b', -9]) == ['a', -5, 'b', -9]  -- identity holds on both sides
combOp(zero, ['a', 0, 'b', 0]) == ['a', 0, 'b', 0]  -- identity holds on both sides


---

## 4. The distributed version

One call. One job. Per-partition folding on the executors, tree-combined partials, a four-element
answer at the driver.

In [7]:
# Find the taxi vehicle with the maximum and the taxi vehicle with the minimum number of trips.
# The md5sum of the taxi's identifier is used as the vehicle identifier.
r1 = df.groupBy("medallion").count().rdd.map(tuple)
by_vehicle = r1.treeAggregate(agg_zero_val, seqOp, combOp)

print("busiest vehicle :", by_vehicle[0], "with", by_vehicle[1], "trips")
print("quietest vehicle:", by_vehicle[2], "with", by_vehicle[3], "trips")
print("\nraw accumulator:", by_vehicle)

busiest vehicle : 696321779D687411F2E5DF6991E9D474 with 7 trips
quietest vehicle: 6D9C2E4EAC8F6A5C7D3102177BC42C03 with 1 trips

raw accumulator: ['696321779D687411F2E5DF6991E9D474', 7, '6D9C2E4EAC8F6A5C7D3102177BC42C03', 1]


In [8]:
# The same question about drivers rather than vehicles.
# The md5sum of the taxi license (driver ID) is used as the driver identifier.
r2 = df.groupBy("hack_license").count().rdd.map(tuple)
by_driver = r2.treeAggregate(agg_zero_val, seqOp, combOp)

print("busiest driver :", by_driver[0], "with", by_driver[1], "trips")
print("quietest driver:", by_driver[2], "with", by_driver[3], "trips")
print("\nraw accumulator:", by_driver)

busiest driver : 00B7691D86D96AEBD21DD9E138F90840 with 10 trips
quietest driver: D8B109DC861AA892745CC4CFF78D98E3 with 1 trips

raw accumulator: ['00B7691D86D96AEBD21DD9E138F90840', 10, 'D8B109DC861AA892745CC4CFF78D98E3', 1]


The two answers differ because the two questions differ: the first is about the busiest and least
busy **taxi**, the second about the busiest and least busy **driver**. The same accumulator and the
same two functions answer both — which is the point of writing the aggregation in terms of
`seqOp` and `combOp` rather than in terms of taxis.

---

## 5. The same aggregation, by hand

What the `treeAggregate` call hides is worth making explicit. The cells below collect the same
`(key, count)` pairs to the driver and re-enact the aggregation in plain Python, with the data
split into artificial partitions of one thousand records.

The simulation is single-machine **on purpose**: it collects the data, deliberately leaving the
distributed world, exactly so that the two-level structure can be laid out whole.

In [9]:
r = df.groupBy("hack_license").count().rdd.map(tuple)
data = r.collect()                      # now local
print("pairs collected:", len(data))
print("first three    :", data[:3])

pairs collected: 4628
first three    : [('130328475AD7427AFDE50A846CA08B22', 1), ('D4F2AE0988ECB2E421AAC0C876483801', 3), ('DD97899ACAC51EF3188A659DB1F4EDBB', 5)]


In [10]:
partition_size = 1000
partitions = [data[i:i + partition_size] for i in range(0, len(data), partition_size)]

print("partitions:", len(partitions))
print("sizes     :", [len(p) for p in partitions])

partitions: 5
sizes     : [1000, 1000, 1000, 1000, 628]


### The executors

The outer loop stands for the executors: each folds its **own** partition's records through
`seqOp`, independently and in parallel, and knows nothing about the others.

In [11]:
# The list for storing the partitions aggregates.
partitions_rez = []
for partition in partitions:
    # This code will be executed on each worker node
    part_rez = agg_zero_val              # initialise the aggregate
    for element in partition:
        part_rez = seqOp(part_rez, element)   # update the aggregate with each element
    # Store the final partition aggregate in the list (for our simulation)
    # In Spark, at this point, the part_rez will be sent to the master node
    partitions_rez.append(part_rez)

for i, rez in enumerate(partitions_rez):
    print(i, rez)

0 ['529E7364F15A734DFB3F44376DC78267', 6, '3E8783B29ABF15A4F5B475D928D2A7BD', 1]
1 ['14C2ED390669165F2D3B5CE427A91027', 7, '2FBF9DAD51F548A6F1CE63165CC30BA7', 1]
2 ['DCD6A3DA3488EF99AAC46FE0EF41449B', 5, 'A8B92DC57A3DF887FF41DA5EBA763A97', 1]
3 ['00B7691D86D96AEBD21DD9E138F90840', 10, '956BC98C042F35EB171FF7FB1A6721B9', 1]
4 ['35A32952035266841AED93F2EFCEEB9D', 6, 'D8B109DC861AA892745CC4CFF78D98E3', 1]


### The driver

The final loop stands for the driver, merging the per-partition results through `combOp`.

It starts from the **first partition's aggregate**, not from the zero value: the zero value was
already folded into every partition by the loop above, and `combOp` merges accumulators. (Because
the zero value is an identity, starting from it instead would give the same answer — which is
precisely what being an identity means.)

In [12]:
final_rez = partitions_rez[0]
# In the simulation, we use a loop to process all partitions' aggregates.
for part_rez in partitions_rez[1:]:
    # In Spark, the master node will receive aggregates from the worker nodes
    # and will execute the combOp() function
    final_rez = combOp(final_rez, part_rez)

print(final_rez)

['00B7691D86D96AEBD21DD9E138F90840', 10, 'D8B109DC861AA892745CC4CFF78D98E3', 1]


### The two must agree

The hand simulation and `treeAggregate` were given the same records, the same zero value and the
same two functions. If the reasoning above is right they must produce the same accumulator, and an
assertion that runs is worth more than a sentence claiming it does.

In [13]:
print("treeAggregate :", by_driver)
print("simulation    :", final_rez)
assert final_rez == by_driver
print("\nidentical.")

# and the grouping really is free: starting the merge from the zero value changes nothing
from functools import reduce
from_zero = reduce(combOp, partitions_rez, agg_zero_val)
assert from_zero == final_rez
print("merging from the zero value instead:", from_zero)

treeAggregate : ['00B7691D86D96AEBD21DD9E138F90840', 10, 'D8B109DC861AA892745CC4CFF78D98E3', 1]
simulation    : ['00B7691D86D96AEBD21DD9E138F90840', 10, 'D8B109DC861AA892745CC4CFF78D98E3', 1]

identical.
merging from the zero value instead: ['00B7691D86D96AEBD21DD9E138F90840', 10, 'D8B109DC861AA892745CC4CFF78D98E3', 1]


---

## Conclusion

The comments in the original notebook speak of worker nodes and a master node; in this book's
vocabulary those are the **executors** and the **driver**.

Its two loops are the two levels of every distributed aggregation:

* the **outer loop over partitions** stands for the executors, each folding its own partition's
  records through `seqOp` independently and in parallel;
* the **final loop** stands for the driver, merging the per-partition results through `combOp`.

An iterative fit is this aggregation **run in a loop**: the current parameter vector travels from
the driver to every executor as a broadcast variable, each executor computes its partition's
contribution — the per-partition `seqOp` — the partials are combined associatively, in a tree when
they are large, and the driver applies the update and begins the next iteration. The driver is
inside the loop in a way it never is for a single-node fit, and every one of those steps is
network traffic paid **once per iteration**. That is why the training data must be cached: without
it, every iteration re-derives the data from its source.

**Exercise 9** extends `seqOp` and `combOp` so that the same single pass also returns the total
number of trips and the number of distinct keys seen, so that the driver can report the mean trips
per taxi. The identity check in section 3 is the part of that exercise most easily got wrong.

In [14]:
spark.stop()